### UnderCover R&D

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

### Connect to LLM Provider

In [ ]:
load_dotenv(override=True)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/" 

if anthropic_api_key:
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")

anthropic = OpenAI(base_url=ANTHROPIC_BASE_URL, api_key=anthropic_api_key, timeout=120, max_retries=5)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

response = anthropic.chat.completions.create(model="claude-haiku-5-5", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
DATA = Path("../../data")

def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/claims/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/claims/held_out_decisions.json" if held_out else "evals/claims/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

An example of a claim:

In [ ]:
claims[0]

And that claims decision:

In [ ]:
labels[claims[0]["claim_id"]]

## Explore the data

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

This chart counts the claims of each type, split by the handler's decision.

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Breakdown by difficulty:

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()